## Chapter 9 Exercises

### 9.1

In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

In [2]:
# Updated from Exercise 8.1
def getTestData(
    n_features=10,
    n_informative=5,
    n_redundant=0,
    n_samples=1000
):
    # Generate synthetic classification data
    trnsX, cont = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        n_redundant=n_redundant,
        n_repeated=0,
        random_state=0,
        shuffle=False
    )

    # Business-day datetime index
    idx = pd.date_range(
        end=pd.Timestamp.today(),
        periods=n_samples,
        freq=pd.tseries.offsets.BDay()
    )

    trnsX = pd.DataFrame(trnsX, index=idx)
    cont = pd.Series(cont, index=idx).to_frame("bin")

    # Feature names: 5 informative, 5 noise
    cols = (
        ["I_" + str(i) for i in range(n_informative)] +
        ["N_" + str(i) for i in range(n_features - n_informative)]
    )

    trnsX.columns = cols

    cont["w"] = 1.0 / cont.shape[0]
    cont["t1"] = pd.Series(cont.index, index=cont.index)

    return trnsX, cont

In [4]:
# (a)
# Create synthetic dataset
trnsX, cont = getTestData(
    n_features=10,
    n_informative=5,
    n_redundant=0,
    n_samples=1000
)

# SVC with RBF kernel
svc = SVC(kernel="rbf", probability=True, random_state=0)

param_grid = {"C": [1E2, 1E-1, 1, 10, 100], "gamma": [1E-2, 1E-1, 1, 10, 100]}

clf = GridSearchCV(estimator=svc, param_grid=param_grid, scoring="neg_log_loss", cv=10, n_jobs=-1)

start_time = time.time()

clf.fit(trnsX, cont["bin"], sample_weight=cont["w"])

end_time = time.time()

elapsed_time = end_time - start_time

In [6]:
# (b)
print(" 25 Nodes")

# (c) 
print("250 Fits")

# (d)
print(f"Elapsed time: {elapsed_time:.2f} seconds")

# (e)
print("Best parameters:", clf.best_params_)
print("Best CV score:", clf.best_score_)
print(pd.DataFrame(clf.cv_results_))

# (f)
print("Best log loss:", -clf.best_score_)
print("Best estimator:", clf.best_estimator_)


# (g)
print(clf.fit(trnsX, cont["bin"], sample_weight=cont["w"]))

 25 Nodes
250 Fits
Elapsed time: 13.52 seconds
Best parameters: {'C': 100.0, 'gamma': 0.1}
Best CV score: -0.17685504141449082
    mean_fit_time  std_fit_time  mean_score_time  std_score_time  param_C  \
0        0.378131      0.158223         0.026800        0.011511    100.0   
1        0.328476      0.119025         0.018576        0.005383    100.0   
2        0.469326      0.075689         0.023468        0.008993    100.0   
3        0.465666      0.103151         0.025414        0.005300    100.0   
4        0.741962      0.361543         0.033979        0.018183    100.0   
5        0.372704      0.078171         0.019583        0.006492      0.1   
6        0.312183      0.069298         0.019053        0.006695      0.1   
7        0.374427      0.099490         0.022034        0.004749      0.1   
8        0.412579      0.122007         0.023710        0.015523      0.1   
9        0.942904      0.608768         0.034180        0.023572      0.1   
10       0.384047      0.0